# 01 — Simulator sanity check

Exploratory only (never imported by `src/`). Builds the nominal vehicle, draws a
truth model, flies a short scenario closed-loop, replays variant `V0` against the
same recorded controls, and plots altitude and airspeed so a human can eyeball that
the simulator produces a sane, bounded trajectory (sim spec §10, Phase 1 DoD).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from aerovalid.sim.maneuvers import load_scenario_yaml
from aerovalid.sim.protocols import State, load_vehicle_params
from aerovalid.sim.runners import controls_from_dataframe, state_from_row
from aerovalid.sim.truth import TruthModel
from aerovalid.sim.variants import load_variant_yaml, make_variant

In [ ]:
nominal = load_vehicle_params("generic_uav_v1", REPO_ROOT / "configs" / "vehicles")
init = State(t_s=0.0, x_m=0.0, y_m=0.0, h_m=1500.0, V_mps=30.0, gamma_rad=0.0, chi_rad=0.0, m_kg=nominal.mass_nom_kg)
scenario = load_scenario_yaml("climb_cruise", REPO_ROOT / "configs" / "scenarios", init)

truth = TruthModel.from_nominal(nominal, master_seed=20260101)
truth_result = truth.run(scenario, seed=1)
print("truth status:", truth_result.status, "| rows:", len(truth_result.trajectory))
truth_result.trajectory.head()

In [ ]:
v0_spec = load_variant_yaml("V0", REPO_ROOT / "configs" / "variants")
v0 = make_variant(v0_spec, nominal)
controls = controls_from_dataframe(truth_result.trajectory)
v0_result = v0.run(None, controls, state_from_row(truth_result.trajectory.iloc[0]), seed=0)
print("V0 status:", v0_result.status, "| rows:", len(v0_result.trajectory))

In [ ]:
fig, (ax_alt, ax_speed) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax_alt.plot(truth_result.trajectory["t_s"], truth_result.trajectory["alt_m"], label="truth")
ax_alt.plot(v0_result.trajectory["t_s"], v0_result.trajectory["alt_m"], label="V0 (replay)", linestyle="--")
ax_alt.set_ylabel("altitude [m]")
ax_alt.legend()

ax_speed.plot(truth_result.trajectory["t_s"], truth_result.trajectory["vel_air_mps"], label="truth")
ax_speed.plot(v0_result.trajectory["t_s"], v0_result.trajectory["vel_air_mps"], label="V0 (replay)", linestyle="--")
ax_speed.set_ylabel("airspeed [m/s]")
ax_speed.set_xlabel("time [s]")

fig.suptitle("climb_cruise: truth vs V0 replay")
fig.tight_layout()
plt.show()

In [ ]:
assert np.all(np.isfinite(truth_result.trajectory.select_dtypes(include="number").to_numpy()))
assert np.all(np.isfinite(v0_result.trajectory.select_dtypes(include="number").to_numpy()))
print("sanity check passed: no NaN/inf in either trajectory")